In [0]:
from pyspark.sql import functions as F

df_est_gold = spark.table("workspace.default.gold_dp_cnpj_estabelecimentos_360")
df_long_gold = spark.table("workspace.default.gold_dp_risco_longevidade_empresarial")
df_qsa_gold = spark.table("workspace.default.gold_dp_rede_societaria_qsa")

# Agrega métricas societárias avançadas
df_soc_kpis = (
    df_qsa_gold
    .groupBy("nr_cnpj_basico")
    .agg(
        F.count("id_socio_hash").alias("qt_total_socios"),
        F.sum(F.when(F.col("tp_socio") == "PESSOA JURIDICA", 1).otherwise(0)).alias("qt_socios_pj"),
        F.sum(F.when(F.col("tp_socio") == "ESTRANGEIRO", 1).otherwise(0)).alias("qt_socios_estrangeiros")
    )
)

df_score_saude = (
    df_est_gold.alias("est")
    .join(df_long_gold.alias("lon"), "nr_cnpj", "inner")
    .join(df_soc_kpis.alias("soc"), "nr_cnpj_basico", "left")
    .select(
        F.col("est.nr_cnpj"),
        F.col("est.nr_cnpj_basico"),
        F.col("est.nm_razao_social"),
        F.col("est.tp_estabelecimento"),
        F.col("est.ds_situacao_cadastral"),
        F.col("lon.nu_idade_empresa_anos"),
        F.col("est.vl_capital_social"),
        F.col("est.fl_simples_nacional_ativo"),
        F.col("est.fl_mei_ativo"),
        F.coalesce(F.col("soc.qt_total_socios"), F.lit(0)).alias("qt_total_socios"),
        # Complexidade de governança/quadro de sócios
        F.when((F.col("soc.qt_socios_estrangeiros") > 0) | (F.col("soc.qt_socios_pj") > 1), "ALTA")
         .when(F.col("soc.qt_total_socios") > 3, "MEDIA")
         .otherwise("BAIXA").alias("ds_complexidade_societaria"),
        # Indicador de alerta de risco / conformidade
        F.when(
            (F.col("est.ds_situacao_cadastral") != "ATIVA"),
            True
        ).otherwise(False).alias("fl_risco_cadastral_elevado"),
        # Segmentação Estratégica
        F.when(F.col("est.fl_mei_ativo") == True, "MICROEMPREENDEDOR_MEI")
         .when((F.col("lon.nu_idade_empresa_anos") >= 10) & (F.col("est.vl_capital_social") >= 500000), "CORPORATE_ESTABELECIDA")
         .when(F.col("lon.nu_idade_empresa_anos") < 2, "EMPRESA_RECENTE_STARTUP")
         .otherwise("PYME_PADRAO").alias("ds_segmento_operacional"),
        F.col("est.sg_uf"),
        F.current_timestamp().alias("dt_carga_gold")
    )
)

(
    df_score_saude.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("sg_uf")
    .saveAsTable("workspace.default.gold_dp_score_saude_e_propensao_pj")
)
print("-> workspace.default.gold_dp_score_saude_e_propensao_pj gravada com sucesso.")

-> workspace.default.gold_dp_score_saude_e_propensao_pj gravada com sucesso.
